# TürKod için Kaggle'da Qwen: Kodi'nin beyni

Bu not defteri Kaggle'ın ücretsiz GPU'sunda **Qwen3 (8B)** modelini çalıştırır ve TürKod'un
bağlanabileceği bir adres verir. Kodi ve AI Asistan bu modelle **token sınırı olmadan** konuşur.

## Bir kez yapılacaklar
1. Kaggle hesabının **telefon doğrulaması** yapılmış olmalı (GPU ve internet için gerekir).
2. Kaggle → **Create → New Notebook** → **File → Import Notebook** → bu dosyayı seç.
3. Sağdaki **Settings** bölümünde: **Accelerator: GPU T4 x2** (ya da P100), **Internet: On**.
4. *(İsteğe bağlı, önerilir)* **Sabit adres** için [ngrok.com](https://ngrok.com)'da ücretsiz hesap aç:
   - **Your Authtoken** sayfasındaki anahtarı kopyala.
   - **Domains** sayfasından ücretsiz sabit alan adını al (ör. `kodi-beyni.ngrok-free.app`).
   - Kaggle'da **Add-ons → Secrets**: `NGROK_TOKEN` (anahtar) ve `NGROK_DOMAIN` (alan adı) ekle,
     bu not defterine bağla.

   ngrok yoksa cloudflared kullanılır: her çalıştırmada **yeni** bir adres verir, TürKod'daki adresi
   her seferinde değiştirmen gerekir.

## Her kullanımda
- **Run All**. İlk çalıştırma ~5-8 dakika sürer (Ollama kurulumu + ~5 GB model indirme).
- Son hücreden önceki hücre TürKod'a yazılacak **adresi** ve **model adını** gösterir.
- Sabit adres (ngrok) kullanıyorsan **Save Version → Save & Run All** ile arka planda da
  çalıştırabilirsin: tarayıcıyı kapatsan da ~12 saate kadar açık kalır.

## TürKod'da
1. **Ayarlar → Yapay Zekâ: Modeller → Sağlayıcı bağla** → **Özel (OpenAI uyumlu adres)**.
2. Adres: hücrenin gösterdiği `https://.../v1` adresi. Anahtar boş kalabilir.
3. Model: `qwen3-turkod` (listede çıkar ya da elle yaz) → kaydet.
4. **Ayarlar → Yapay Zekâ: Görevler** → **Kodi** (ve istersen AI Paneli sohbet) → bu model.

Groq modellerini de bağlı tut: Kaggle oturumu kapanınca TürKod beklemeden Groq'a geçer, oturum
yeniden açılınca kendiliğinden Kaggle'a döner.

## Bilmen gerekenler
- Kaggle'ın GPU kotası haftada yaklaşık 30 saat, bir oturum en çok ~12 saat (güncel değer için
  Kaggle'daki kota göstergesine bak). Kullanmadığında oturumu **durdur**, kota boşa gitmesin.
- Adresi bilen herkes modeli kullanabilir: adresi kimseyle paylaşma.
- Kaggle not defterlerini sürekli açık bir sunucu gibi kullanmak Kaggle kurallarında gri bir
  alandır; oturumun Kaggle tarafından kapatılması mümkündür.

In [ ]:
# Ayarlar (istersen değiştir)
MODEL = "qwen3:8b"      # Ollama model adı. Daha güçlüsü: "qwen3:14b" (daha yavaş, T4'e sığar)
AD = "qwen3-turkod"     # TürKod'a yazılacak model adı ("qwen3" ile başlamalı: TürKod düşünmeyi
                        # bu adla tanıyıp kapatır, Kodi hızlı cevap verir)
BAGLAM = 16384          # Kodi'nin istekleri ~5-6 bin token; 16k rahat yeter

In [ ]:
# 1) Ollama'yı kur ve başlat
import os, re, subprocess, time, requests

subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null",
               shell=True, check=False)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

os.environ.update(
    OLLAMA_HOST="0.0.0.0:11434",     # tünel dışarıdan ulaşabilsin
    OLLAMA_ORIGINS="*",
    OLLAMA_KEEP_ALIVE="-1",          # model GPU'dan hiç boşaltılmasın (ilk cevap beklemesin)
    OLLAMA_CONTEXT_LENGTH=str(BAGLAM),
    OLLAMA_NUM_PARALLEL="1",
)
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Ollama hazır:", requests.get("http://127.0.0.1:11434/api/version", timeout=2).json())
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("Ollama başlamadı; /kaggle/working/ollama.log dosyasına bak.")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

In [ ]:
# 2) Modeli indir ve TürKod için ayarla (bağlam: BAGLAM token)
subprocess.run(["ollama", "pull", MODEL], check=True)
with open("/kaggle/working/Modelfile", "w") as f:
    f.write(f"FROM {MODEL}\nPARAMETER num_ctx {BAGLAM}\n")
subprocess.run(["ollama", "create", AD, "-f", "/kaggle/working/Modelfile"], check=True)
print("Hazır model:", AD)

In [ ]:
# 3) Yerelde dene: model GPU'ya yüklenir, Kodi'nin "eylem" aracını çağırabiliyor mu?
ARAC = [{"type": "function", "function": {
    "name": "eylem", "description": "TürKod'da bir iş yapar",
    "parameters": {"type": "object", "properties": {
        "eylem": {"type": "string"}, "panel": {"type": "string"}, "durum": {"type": "string"}}}}}]
t0 = time.time()
r = requests.post("http://127.0.0.1:11434/v1/chat/completions", timeout=900, json={
    "model": AD, "max_tokens": 300, "tools": ARAC,
    "messages": [
        {"role": "system", "content": 'Sen TürKod\'daki yardımcı robotsun. Terminal panelini açmak '
                                      'için eylem aracını {"eylem": "panel", "panel": "terminal", '
                                      '"durum": "ac"} ile çağır.'},
        {"role": "user", "content": "Terminali açar mısın? /no_think"}]}).json()
mesaj = r["choices"][0]["message"]
print(f"{time.time() - t0:.1f} sn (ilk istek modeli GPU'ya yükler, sonrakiler hızlıdır)")
print("Yazı:", (mesaj.get("content") or "").strip()[:300])
print("Araç çağrısı:", mesaj.get("tool_calls"))
subprocess.run(["ollama", "ps"])   # PROCESSOR sütununda "100% GPU" görmelisin

In [ ]:
# 4) Dışarıya aç: ngrok (Secrets'ta NGROK_TOKEN varsa, sabit adres) ya da cloudflared
def gizli(ad):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(ad) or ""
    except Exception:
        return ""

ngrok_anahtari, ngrok_alani = gizli("NGROK_TOKEN"), gizli("NGROK_DOMAIN")
if ngrok_anahtari:
    subprocess.run("pip -q install pyngrok", shell=True, check=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_anahtari)
    secenek = {"domain": ngrok_alani} if ngrok_alani else {}
    ADRES = ngrok.connect(11434, "http", host_header="rewrite", **secenek).public_url
else:
    subprocess.run("wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/"
                   "releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared",
                   shell=True, check=True)
    tunel = subprocess.Popen(
        ["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:11434",
         "--http-host-header", "localhost:11434"],
        stdout=open("/kaggle/working/cloudflared.log", "w"), stderr=subprocess.STDOUT)
    ADRES = ""
    for _ in range(90):
        bulunan = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                            open("/kaggle/working/cloudflared.log").read())
        if bulunan:
            ADRES = bulunan.group(0)
            break
        time.sleep(1)
    if not ADRES:
        raise SystemExit("Tünel açılmadı; /kaggle/working/cloudflared.log dosyasına bak.")
    time.sleep(8)   # yeni adresin internette duyulması birkaç saniye sürer

for _ in range(10):
    try:
        modeller = requests.get(ADRES + "/v1/models", timeout=20).json()
        break
    except Exception as hata:
        print("Adres henüz hazır değil, yeniden deneniyor:", hata)
        time.sleep(5)
print("Adresten görünen modeller:", [m["id"] for m in modeller.get("data", [])])
print()
print("=" * 60)
print("TürKod → Ayarlar → Yapay Zekâ: Modeller → Sağlayıcı bağla → Özel (OpenAI uyumlu adres)")
print("  Adres :", ADRES + "/v1")
print("  Model :", AD)
print("=" * 60)

In [ ]:
# 5) Oturumu açık tut (bu hücre çalıştıkça model hizmette). Bitirince hücreyi durdur ve
#    sağ üstten oturumu kapat: GPU kotan boşa gitmesin.
while True:
    time.sleep(300)
    try:
        durum = requests.get(ADRES + "/v1/models", timeout=30).status_code
    except Exception as hata:
        durum = f"ulaşılamadı ({hata.__class__.__name__})"
    if ollama.poll() is not None:
        durum = "Ollama durdu! /kaggle/working/ollama.log dosyasına bak."
    print(time.strftime("%H:%M"), "adres:", durum, flush=True)